# Cell 1 — Import library

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Cell 2 — Folder project

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_MODELS_DIR = PROJECT_ROOT / "outputs" / "models"

OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", PROCESSED_DIR)
print("Output tables dir:", OUTPUT_TABLES_DIR)
print("Output models dir:", OUTPUT_MODELS_DIR)

Project root: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL
Processed dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\data\processed
Output tables dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables
Output models dir: e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\models


# Cell 3 — Baca dataset train/test dan feature set

In [3]:
TRAIN_DATASET_PATH = PROCESSED_DIR / "10_train_dataset.csv"
TEST_DATASET_PATH = PROCESSED_DIR / "10_test_dataset.csv"
FEATURE_SET_JSON_PATH = PROCESSED_DIR / "10_feature_sets.json"

train_df = pd.read_csv(TRAIN_DATASET_PATH)
test_df = pd.read_csv(TEST_DATASET_PATH)

with open(FEATURE_SET_JSON_PATH, "r", encoding="utf-8") as f:
    feature_sets = json.load(f)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

print("\nDistribusi train:")
display(train_df["label_performance"].value_counts())

print("\nDistribusi test:")
display(test_df["label_performance"].value_counts())

print("\nFeature sets:")
for name, features in feature_sets.items():
    print(name, "=", len(features), "fitur")

Train: (71, 49)
Test: (18, 49)

Distribusi train:


label_performance
Sedang    43
Tinggi    16
Rendah    12
Name: count, dtype: int64


Distribusi test:


label_performance
Sedang    11
Tinggi     4
Rendah     3
Name: count, dtype: int64


Feature sets:
all_process_features = 47 fitur
core_behavior_features = 20 fitur
activity_specific_features = 13 fitur
transition_specific_features = 14 fitur
frequency_and_ratio_features = 10 fitur


# Cell 4 — Definisikan model

In [4]:
models = {
    "Dummy_Most_Frequent": DummyClassifier(strategy="most_frequent"),

    "Logistic_Regression_Balanced": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random_Forest_Balanced": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        class_weight="balanced",
        random_state=42
    )
}

print("Model yang akan diuji:")
for model_name in models:
    print("-", model_name)

Model yang akan diuji:
- Dummy_Most_Frequent
- Logistic_Regression_Balanced
- Random_Forest_Balanced


# Cell 5 — Fungsi evaluasi model

In [5]:
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    result = {
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "test_macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "test_macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0)
    }

    return result, y_pred, model

# Cell 6 — Fungsi cross-validation

In [17]:
from sklearn.metrics import make_scorer

def cross_validate_model(model, X, y):
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    scoring = {
        "accuracy": "accuracy",
        "macro_precision": make_scorer(
            precision_score,
            average="macro",
            zero_division=0
        ),
        "macro_recall": make_scorer(
            recall_score,
            average="macro",
            zero_division=0
        ),
        "macro_f1": make_scorer(
            f1_score,
            average="macro",
            zero_division=0
        )
    }

    cv_result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        error_score="raise"
    )

    return {
        "cv_accuracy_mean": cv_result["test_accuracy"].mean(),
        "cv_accuracy_std": cv_result["test_accuracy"].std(),
        "cv_macro_precision_mean": cv_result["test_macro_precision"].mean(),
        "cv_macro_recall_mean": cv_result["test_macro_recall"].mean(),
        "cv_macro_f1_mean": cv_result["test_macro_f1"].mean(),
        "cv_macro_f1_std": cv_result["test_macro_f1"].std()
    }

# Cell 7 — Training semua model dan feature set

In [18]:
target_col = "label_performance"

evaluation_rows = []
prediction_records = {}
trained_models = {}

for feature_set_name, features in feature_sets.items():
    print("\nFeature set:", feature_set_name)
    print("Jumlah fitur:", len(features))

    X_train = train_df[features]
    y_train = train_df[target_col]

    X_test = test_df[features]
    y_test = test_df[target_col]

    for model_name, model in models.items():
        print("  Training:", model_name)

        cv_metrics = cross_validate_model(model, X_train, y_train)
        test_metrics, y_pred, fitted_model = evaluate_model(
            model,
            X_train,
            y_train,
            X_test,
            y_test
        )

        row = {
            "feature_set": feature_set_name,
            "model": model_name,
            "jumlah_fitur": len(features),
            **cv_metrics,
            **test_metrics
        }

        evaluation_rows.append(row)

        key = f"{feature_set_name}__{model_name}"
        prediction_records[key] = {
            "y_test": y_test,
            "y_pred": y_pred
        }
        trained_models[key] = fitted_model

evaluation_results = pd.DataFrame(evaluation_rows)

evaluation_results = evaluation_results.sort_values(
    by=["test_macro_f1", "test_accuracy"],
    ascending=False
).reset_index(drop=True)

display(evaluation_results)


Feature set: all_process_features
Jumlah fitur: 47
  Training: Dummy_Most_Frequent
  Training: Logistic_Regression_Balanced
  Training: Random_Forest_Balanced

Feature set: core_behavior_features
Jumlah fitur: 20
  Training: Dummy_Most_Frequent
  Training: Logistic_Regression_Balanced
  Training: Random_Forest_Balanced

Feature set: activity_specific_features
Jumlah fitur: 13
  Training: Dummy_Most_Frequent
  Training: Logistic_Regression_Balanced
  Training: Random_Forest_Balanced

Feature set: transition_specific_features
Jumlah fitur: 14
  Training: Dummy_Most_Frequent
  Training: Logistic_Regression_Balanced
  Training: Random_Forest_Balanced

Feature set: frequency_and_ratio_features
Jumlah fitur: 10
  Training: Dummy_Most_Frequent
  Training: Logistic_Regression_Balanced
  Training: Random_Forest_Balanced


,feature_set,model,jumlah_fitur,cv_accuracy_mean,cv_accuracy_std,cv_macro_precision_mean,cv_macro_recall_mean,cv_macro_f1_mean,cv_macro_f1_std,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1
0,all_process_features,Random_Forest_Balanced,47,0.620952,0.149424,0.567619,0.607407,0.554585,0.177150,0.722222,0.750000,0.848485,0.743231
1,core_behavior_features,Random_Forest_Balanced,20,0.494286,0.070912,0.427381,0.515741,0.436835,0.133236,0.722222,0.691667,0.795455,0.717836
2,transition_specific_features,Logistic_Regression_Balanced,14,0.480000,0.087877,0.392593,0.494444,0.417747,0.082420,0.666667,0.652381,0.765152,0.672222
3,frequency_and_ratio_features,Random_Forest_Balanced,10,0.480952,0.111066,0.435224,0.483333,0.435507,0.142265,0.666667,0.652381,0.765152,0.672222
4,activity_specific_features,Random_Forest_Balanced,13,0.649524,0.112066,0.588889,0.638889,0.594043,0.163538,0.611111,0.652778,0.734848,0.648459
5,transition_specific_features,Random_Forest_Balanced,14,0.634286,0.136606,0.578803,0.637963,0.577933,0.168342,0.611111,0.652778,0.734848,0.648459
6,activity_specific_features,Logistic_Regression_Balanced,13,0.435238,0.056311,0.366422,0.437963,0.355902,0.078825,0.611111,0.666667,0.787879,0.622222
7,core_behavior_features,Logistic_Regression_Balanced,20,0.535238,0.095836,0.468889,0.552778,0.471625,0.136664,0.555556,0.591667,0.704545,0.570707
8,frequency_and_ratio_features,Logistic_Regression_Balanced,10,0.551429,0.102857,0.526852,0.570370,0.508052,0.126879,0.555556,0.576190,0.704545,0.566667
9,all_process_features,Logistic_Regression_Balanced,47,0.448571,0.144165,0.348360,0.434259,0.368163,0.124159,0.555556,0.538095,0.623737,0.551852


# Cell 8 — Simpan hasil evaluasi

In [19]:
EVALUATION_RESULTS_PATH = OUTPUT_TABLES_DIR / "11_model_evaluation_results.csv"

evaluation_results.to_csv(EVALUATION_RESULTS_PATH, index=False)

print("Hasil evaluasi model disimpan ke:")
print(EVALUATION_RESULTS_PATH)

Hasil evaluasi model disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\11_model_evaluation_results.csv


# Cell 9 — Lihat model terbaik

In [20]:
best_row = evaluation_results.iloc[0]

best_feature_set = best_row["feature_set"]
best_model_name = best_row["model"]
best_key = f"{best_feature_set}__{best_model_name}"

print("Model terbaik berdasarkan test_macro_f1:")
display(best_row)

print("\nBest key:", best_key)

Model terbaik berdasarkan test_macro_f1:


feature_set                  all_process_features
model                      Random_Forest_Balanced
jumlah_fitur                                   47
cv_accuracy_mean                         0.620952
cv_accuracy_std                          0.149424
cv_macro_precision_mean                  0.567619
cv_macro_recall_mean                     0.607407
cv_macro_f1_mean                         0.554585
cv_macro_f1_std                           0.17715
test_accuracy                            0.722222
test_macro_precision                         0.75
test_macro_recall                        0.848485
test_macro_f1                            0.743231
Name: 0, dtype: object


Best key: all_process_features__Random_Forest_Balanced


# Cell 10 — Classification report model terbaik

In [21]:
best_predictions = prediction_records[best_key]

y_test_best = best_predictions["y_test"]
y_pred_best = best_predictions["y_pred"]

print("Classification report model terbaik:")
print(classification_report(
    y_test_best,
    y_pred_best,
    zero_division=0
))

Classification report model terbaik:
              precision    recall  f1-score   support

      Rendah       0.75      1.00      0.86         3
      Sedang       1.00      0.55      0.71        11
      Tinggi       0.50      1.00      0.67         4

    accuracy                           0.72        18
   macro avg       0.75      0.85      0.74        18
weighted avg       0.85      0.72      0.72        18



# Cell 11 — Confusion matrix model terbaik

In [22]:
labels_order = ["Rendah", "Sedang", "Tinggi"]

cm = confusion_matrix(
    y_test_best,
    y_pred_best,
    labels=labels_order
)

confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"Actual_{label}" for label in labels_order],
    columns=[f"Predicted_{label}" for label in labels_order]
)

display(confusion_matrix_df)

CONFUSION_MATRIX_PATH = OUTPUT_TABLES_DIR / "11_best_model_confusion_matrix.csv"
confusion_matrix_df.to_csv(CONFUSION_MATRIX_PATH)

print("Confusion matrix disimpan ke:")
print(CONFUSION_MATRIX_PATH)

,Predicted_Rendah,Predicted_Sedang,Predicted_Tinggi
Actual_Rendah,3,0,0
Actual_Sedang,1,6,4
Actual_Tinggi,0,0,4


Confusion matrix disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\11_best_model_confusion_matrix.csv


# Cell 12 — Simpan classification report ke CSV

In [23]:
report_dict = classification_report(
    y_test_best,
    y_pred_best,
    zero_division=0,
    output_dict=True
)

classification_report_df = pd.DataFrame(report_dict).T

CLASSIFICATION_REPORT_PATH = OUTPUT_TABLES_DIR / "11_best_model_classification_report.csv"
classification_report_df.to_csv(CLASSIFICATION_REPORT_PATH)

display(classification_report_df)

print("Classification report disimpan ke:")
print(CLASSIFICATION_REPORT_PATH)

,precision,recall,f1-score,support
Rendah,0.750000,1.000000,0.857143,3.000000
Sedang,1.000000,0.545455,0.705882,11.000000
Tinggi,0.500000,1.000000,0.666667,4.000000
accuracy,0.722222,0.722222,0.722222,0.722222
macro avg,0.750000,0.848485,0.743231,18.000000
weighted avg,0.847222,0.722222,0.722378,18.000000


Classification report disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\11_best_model_classification_report.csv


# Cell 13 — Feature importance Random Forest terbaik

In [24]:
from sklearn.base import clone

if best_model_name == "Random_Forest_Balanced":
    best_features = feature_sets[best_feature_set]

    X_train_best = train_df[best_features]
    y_train_best = train_df[target_col]

    # Fit ulang model terbaik agar jumlah fitur pasti sesuai
    best_model_for_importance = clone(models[best_model_name])
    best_model_for_importance.fit(X_train_best, y_train_best)

    feature_importance_df = pd.DataFrame({
        "feature": best_features,
        "importance": best_model_for_importance.feature_importances_
    }).sort_values("importance", ascending=False)

    display(feature_importance_df.head(20))

    FEATURE_IMPORTANCE_PATH = OUTPUT_TABLES_DIR / "11_best_model_feature_importance.csv"
    feature_importance_df.to_csv(FEATURE_IMPORTANCE_PATH, index=False)

    print("Feature importance disimpan ke:")
    print(FEATURE_IMPORTANCE_PATH)

else:
    print("Model terbaik bukan Random Forest, feature importance tree-based tidak tersedia.")

,feature,importance
43,tr_auto_saved_to_quiz_viewed,0.066268
25,act_quiz_auto_saved,0.060637
42,tr_quiz_updated_to_auto_saved,0.056980
23,act_quiz_viewed,0.040264
40,tr_quiz_viewed_to_quiz_updated,0.039100
10,freq_quiz,0.037051
5,active_days,0.033299
24,act_quiz_updated,0.030653
9,freq_material,0.030098
33,tr_course_to_material,0.030091


Feature importance disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\11_best_model_feature_importance.csv


# Cell 14 — Koefisien Logistic Regression terbaik

In [26]:
if best_model_name == "Logistic_Regression_Balanced":
    best_model = trained_models[best_key]
    best_features = feature_sets[best_feature_set]

    logistic_model = best_model.named_steps["model"]

    coef_df = pd.DataFrame(
        logistic_model.coef_,
        columns=best_features,
        index=logistic_model.classes_
    )

    display(coef_df)

    COEF_PATH = OUTPUT_TABLES_DIR / "11_best_logistic_regression_coefficients.csv"
    coef_df.to_csv(COEF_PATH)

    print("Koefisien Logistic Regression disimpan ke:")
    print(COEF_PATH)
else:
    print("Model terbaik bukan Logistic Regression.")

Model terbaik bukan Logistic Regression.


# Cell 15 — Simpan ringkasan model terbaik

In [27]:
best_model_summary = pd.DataFrame([{
    "best_feature_set": best_feature_set,
    "best_model": best_model_name,
    "jumlah_fitur": int(best_row["jumlah_fitur"]),
    "test_accuracy": best_row["test_accuracy"],
    "test_macro_precision": best_row["test_macro_precision"],
    "test_macro_recall": best_row["test_macro_recall"],
    "test_macro_f1": best_row["test_macro_f1"],
    "cv_macro_f1_mean": best_row["cv_macro_f1_mean"],
    "cv_macro_f1_std": best_row["cv_macro_f1_std"]
}])

display(best_model_summary)

BEST_MODEL_SUMMARY_PATH = OUTPUT_TABLES_DIR / "11_best_model_summary.csv"
best_model_summary.to_csv(BEST_MODEL_SUMMARY_PATH, index=False)

print("Ringkasan model terbaik disimpan ke:")
print(BEST_MODEL_SUMMARY_PATH)

,best_feature_set,best_model,jumlah_fitur,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,cv_macro_f1_mean,cv_macro_f1_std
0,all_process_features,Random_Forest_Balanced,47,0.722222,0.75,0.848485,0.743231,0.554585,0.17715


Ringkasan model terbaik disimpan ke:
e:\Tugas Akhir\Coding\TA_PROCESS_MINING_FINAL\outputs\tables\11_best_model_summary.csv


## Kesimpulan Model Training dan Evaluation

Pada tahap ini, beberapa model machine learning dilatih menggunakan fitur hasil process mining untuk memprediksi label performa akademik mahasiswa. Model yang diuji meliputi Dummy Classifier sebagai baseline, Logistic Regression Balanced, dan Random Forest Balanced. Evaluasi dilakukan menggunakan beberapa feature set, yaitu all process features, core behavior features, activity specific features, transition specific features, serta frequency and ratio features.

Hasil evaluasi menunjukkan bahwa model terbaik diperoleh dari Random Forest Balanced dengan feature set `all_process_features`. Model ini menggunakan 47 fitur proses dan menghasilkan akurasi sebesar 72,22% serta macro F1-score sebesar 74,32% pada data uji. Hasil ini lebih baik dibandingkan baseline Dummy Classifier yang memperoleh macro F1-score sebesar 25,29%. Dengan demikian, fitur hasil process mining terbukti memberikan informasi yang lebih baik dibandingkan prediksi berdasarkan kelas mayoritas saja.

Berdasarkan classification report, model mampu mengenali seluruh data uji pada kelas Rendah dan Tinggi, tetapi masih mengalami kesalahan pada kelas Sedang. Sebagian mahasiswa dengan label Sedang diprediksi sebagai Tinggi. Hal ini menunjukkan bahwa kelompok Sedang memiliki pola aktivitas yang dapat tumpang tindih dengan kelompok performa lainnya.

Hasil feature importance menunjukkan bahwa fitur yang berkaitan dengan aktivitas kuis dan transisi pengerjaan kuis menjadi fitur yang paling berpengaruh. Beberapa fitur penting meliputi transisi `tr_auto_saved_to_quiz_viewed`, aktivitas `act_quiz_auto_saved`, transisi `tr_quiz_updated_to_auto_saved`, aktivitas `act_quiz_viewed`, transisi `tr_quiz_viewed_to_quiz_updated`, `freq_quiz`, `active_days`, dan `freq_material`. Temuan ini sejalan dengan hasil process mining sebelumnya yang menunjukkan bahwa proses belajar mahasiswa pada LMS didominasi oleh siklus pengerjaan kuis serta didukung oleh akses course dan material.

Meskipun hasil pada data uji menunjukkan performa yang cukup baik, nilai cross-validation macro F1-score masih lebih rendah dan memiliki variasi antar fold. Hal ini menunjukkan bahwa hasil model perlu diinterpretasikan secara hati-hati karena jumlah data mahasiswa relatif terbatas. Oleh karena itu, model digunakan sebagai analisis pendukung untuk memahami kontribusi fitur proses terhadap performa akademik, bukan sebagai sistem prediksi final yang bersifat mutlak.